# Timeseries Analysis Notebook

**NOTE using the de-sar-sample-data envrionment for analysis**

In [ ]:
import fsspec
import xarray as xr
import rioxarray
from dotenv import load_dotenv
import re
import pandas as pd
import numpy as np
from scipy.ndimage import uniform_filter
import geopandas as gpd
from dea_tools.plotting import xr_animation
from PIL import Image, ImageSequence

## Set envrionment credentials for AWS access

In [ ]:
load_dotenv('/home/ec2-user/sar-pipeline/.env')

## Select the burst to assess

In [ ]:
# t007_014550_iw2 (scene 1) - slope and aspect
# t007_014551_iw2 (scene 1) - slope and aspect
# t007_014549_iw1 (scene 1) - elevation
# t007_014543_iw2 (scene 2) - elevation
burst = "t007_014550_iw2"
burst_shape = gpd.read_file(f"burst_shapefiles/{burst}.json")


## Functions

In [ ]:
# Adapted from https://stackoverflow.com/questions/39785970/speckle-lee-filter-in-python
def lee_filter(img, size):
    """
    Applies the Lee filter to reduce speckle noise in an image.

    Parameters:
    img (ndarray): Input image to be filtered.
    size (int): Size of the uniform filter window.

    Returns:
    ndarray: The filtered image.
    """
    img_mean = uniform_filter(img, size)
    img_sqr_mean = uniform_filter(img**2, size)
    img_variance = img_sqr_mean - img_mean**2

    overall_variance = np.var(img)

    img_weights = img_variance / (img_variance + overall_variance)
    img_output = img_mean + img_weights * (img - img_mean)
    return img_output


# Define a function to apply the Lee filter to a DataArray
def apply_lee_filter(data_array, size=7):
    """
    Applies the Lee filter to the provided DataArray.

    Parameters:
    data_array (xarray.DataArray): The data array to be filtered.
    size (int): Size of the uniform filter window. Default is 7.

    Returns:
    xarray.DataArray: The filtered data array.
    """
    data_array_filled = data_array.fillna(0)
    filtered_data = xr.apply_ufunc(
        lee_filter,
        data_array_filled,
        kwargs={"size": size},
        input_core_dims=[["y", "x"]],
        output_core_dims=[["y", "x"]],
        dask_gufunc_kwargs={"allow_rechunk": True},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[data_array.dtype],
    )
    filtered_data_masked = xr.where(np.isnan(data_array), np.nan, filtered_data)

    return filtered_data_masked


def slope_aspect(dem, dx=1.0, dy=1.0):
    """
    Compute slope and aspect from a 2D DEM array, handling nodata values.
    
    Parameters
    ----------
    dem : np.ndarray
        2D elevation array (NaNs represent nodata).
    dx : float
        Spatial resolution in x-direction.
    dy : float
        Spatial resolution in y-direction.
    
    Returns
    -------
    slope : np.ndarray
        Slope in degrees.
    aspect : np.ndarray
        Aspect in degrees (0 = North, clockwise).
    """
    # Mask invalid values
    dem = np.where(np.isfinite(dem), dem, np.nan)
    
    # Compute gradients
    dz_dy, dz_dx = np.gradient(dem, dy, dx)
    
    # Replace NaN gradients with 0 to avoid NaNs in slope/aspect
    dz_dx = np.nan_to_num(dz_dx)
    dz_dy = np.nan_to_num(dz_dy)
    
    # Slope in degrees
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)) * 180 / np.pi
    
    # Aspect in degrees
    aspect = np.arctan2(-dz_dx, -dz_dy) * 180 / np.pi
    aspect = np.where(np.isnan(dem), np.nan, (aspect + 360) % 360)
    
    return slope, aspect


def slope_aspect_timeseries(ds_dem, dx=1.0, dy=1.0):
    """
    Apply slope_aspect function to an xarray DataArray over time,
    preserving NaNs for nodata values.
    
    Parameters
    ----------
    ds_dem : xarray.DataArray
        DEM time series with dimensions ('time', 'y', 'x')
    dx, dy : float
        Spatial resolution
    
    Returns
    -------
    slopes : xarray.DataArray
        Slope for each timestep
    aspects : xarray.DataArray
        Aspect for each timestep
    """
    slopes, aspects = xr.apply_ufunc(
        slope_aspect,
        ds_dem,
        dx,
        dy,
        input_core_dims=[["y", "x"], [], []],
        output_core_dims=[["y", "x"], ["y", "x"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[ds_dem.dtype, ds_dem.dtype],
    )
    
    slopes = xr.DataArray(slopes, coords=ds_dem.coords, dims=ds_dem.dims, name="slope")
    aspects = xr.DataArray(aspects, coords=ds_dem.coords, dims=ds_dem.dims, name="aspect")
    
    return slopes, aspects

def keep_s3_prefix(file_list):
    return [f if f.startswith("s3://") else f"s3://{f}" for f in file_list]

# Extract timestamps from filenames (e.g., 20170405T050842)
def extract_time(fp):
    match = re.search(r"(\d{8}T\d{6})", fp)
    return pd.to_datetime(match.group(1)) if match else None

def load_xr_dataset(file_list):
    datasets = []
    for fp in file_list:
        da = rioxarray.open_rasterio(
            fp,
            masked=True,
            chunks=True,
        )
        da = da.squeeze().expand_dims(time=[extract_time(fp)])  # add time dim
        datasets.append(da)

        # Combine along the time dimension
    return xr.concat(datasets, dim="time")  # dims: ('time', 'y', 'x')

def preprocess_gamma0_xr_dataset(ds_gamma0, burst_shape):
    # Open each file lazily with rioxarray and assign a time coordinate
    ds_gamma0.name = "HH_gamma0"
    # trim the dataset withg burst shapefule
    burst_shape = burst_shape.to_crs(ds_gamma0.rio.crs)
    ds_gamma0 = ds_gamma0.rio.clip(burst_shape.geometry, crs=ds_gamma0.rio.crs, drop=True)
    ds_gamma0.odc.assign_crs(crs='EPSG:3031')
    # Apply Lee filter directly on the DataArray
    ds_gamma0["HH_gamma0_filtered"] = apply_lee_filter(ds_gamma0, size=5)
    # Convert to dB only for positive values
    ds_gamma0["HH_gamma0_db"] = xr.where(
        ds_gamma0.to_dataset(name='HH_gamma0')['HH_gamma0'] > 0,
        10 * np.log10(ds_gamma0.to_dataset(name='HH_gamma0')['HH_gamma0']),
        np.nan
    )
    ds_gamma0["HH_gamma0_filtered_db"] = 10 * np.log10(ds_gamma0.HH_gamma0_filtered)
    return ds_gamma0

def preprocess_dem_xr_dataset(ds_dem, burst_shape):
    ds_dem.name = "dem"
    burst_shape = burst_shape.to_crs(ds_dem.rio.crs)
    ds_dem = ds_dem.rio.clip(burst_shape.geometry, crs=ds_dem.rio.crs, drop=True)
    ds_dem.odc.assign_crs(crs='EPSG:3031')
    ds_dem = ds_dem.to_dataset(name="elevation")
    slopes, aspects = slope_aspect_timeseries(ds_dem['elevation'], dx=30, dy=30)
    ds_dem = ds_dem.assign({
        "slope": slopes,
        "aspect": aspects
    })
    return ds_dem


## Load in the xarrays

In [ ]:
# Create S3 filesystem (anonymous access)
fs = fsspec.filesystem("s3", anon=True)

# Base S3 folder (public)
static_base_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10/ga_s1_nrb_iw_hh_0/{burst}/"
timeseries_base_path = f"s3://deant-data-public-dev/experimental/rema-timeseries-v0.3-assessment/REMA_10_TIMESERIES/ga_s1_nrb_iw_hh_0/{burst}/"

# Find all GeoTIFFs recursively
static_nrb_tif_files = fs.glob(f"{static_base_path}**/*HH-gamma0.tif")
static_dem_tif_files = fs.glob(f"{static_base_path}**/*digital-elevation-model.tif")
timeseries_nrb_tif_files = fs.glob(f"{timeseries_base_path}**/*HH-gamma0.tif")
timeseries_dem_tif_files = fs.glob(f"{timeseries_base_path}**/*digital-elevation-model.tif")

# Ensure each file keeps the s3:// prefix (sometimes fsspec strips it)
static_nrb_tif_files = keep_s3_prefix(static_nrb_tif_files)
static_dem_tif_files = keep_s3_prefix(static_dem_tif_files)
timeseries_nrb_tif_files = keep_s3_prefix(timeseries_nrb_tif_files)
timeseries_dem_tif_files = keep_s3_prefix(timeseries_dem_tif_files)

print(f"Found {len(static_nrb_tif_files)} static NRB TIFs:")
print(f"Found {len(static_dem_tif_files)} static DEM TIFs:")
print(f"Found {len(timeseries_nrb_tif_files)} timeseries NRB TIFs:")
print(f"Found {len(timeseries_dem_tif_files)} timeseries DEM TIFs:")

# load and preprocess the gamma0 datasets
ds_static_nrb = load_xr_dataset(static_nrb_tif_files)
ds_static_nrb = preprocess_gamma0_xr_dataset(ds_static_nrb, burst_shape)
ds_timeseries_nrb = load_xr_dataset(timeseries_nrb_tif_files)
ds_timeseries_nrb = preprocess_gamma0_xr_dataset(ds_timeseries_nrb, burst_shape)

# load in the dem datasets
ds_static_dem = load_xr_dataset(static_dem_tif_files)
ds_static_dem = preprocess_dem_xr_dataset(ds_static_dem, burst_shape)
ds_timeseries_dem = load_xr_dataset(timeseries_dem_tif_files)
ds_timeseries_dem = preprocess_dem_xr_dataset(ds_timeseries_dem, burst_shape)

#ds_static_nrb.isel(time=1).HH_gamma0_filtered_db.plot.imshow(cmap="bone", vmin=-20, vmax=0, figsize=(12,5)) 

In [ ]:
make_animation = True
if make_animation:
    for i,ds in enumerate([ds_static_nrb,ds_timeseries_nrb,ds_static_dem,ds_timeseries_dem]):
        dem_type = ['REMA_10','REMA_10_TIMESERIES','REMA_10','REMA_10_TIMESERIES'][i]
        plot_var = ['HH_gamma0_filtered_db','HH_gamma0_filtered_db','elevation','elevation'][i]
        if plot_var == 'HH_gamma0_filtered_db':
            imshow_kwargs={"cmap":"bone", "vmin":-20, "vmax":0}
            plot_ds = ds[plot_var].to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
            band = ds.name
        if plot_var == 'elevation':
            imshow_kwargs={}
            plot_ds = ds[plot_var].to_dataset(name='elevation').odc.assign_crs(crs='EPSG:3031')
            band = plot_var
        xr_animation(
            plot_ds,
            bands=band,
            output_path=f'{burst}_{dem_type}_{band}.gif',
            width_pixels=1200,
            interval=200,
            show_date='%d %b %Y',
            show_text=f'burst: {burst}  dem_type: {dem_type}',
            show_colorbar=True,
            imshow_kwargs=imshow_kwargs,
            colorbar_kwargs={},
        )

## Animation of NRB differnces at each timestep

In [ ]:
# Compute difference at each timestep
nrb_diff_ts = ds_timeseries_nrb.HH_gamma0_filtered_db - ds_static_nrb.HH_gamma0_filtered_db
# Optionally, wrap as a Dataset
nrb_diff_ds = xr.Dataset({"HH_gamma0_filtered_db_diff": nrb_diff_ts})

In [ ]:
make_animation = True
#.to_dataset(name=ds.name).odc.assign_crs(crs='EPSG:3031')
if make_animation:
    xr_animation(
        nrb_diff_ds.odc.assign_crs(crs='EPSG:3031'),
        bands="HH_gamma0_filtered_db_diff",
        output_path=f'{burst}_gamma0_difference.gif',
        width_pixels=1200,
        interval=200,
        show_date='%d %b %Y',
        show_text=f'burst: {burst}',
        show_colorbar=True,
        imshow_kwargs={"cmap":"RdBu", "vmin":-2, "vmax":2},
        colorbar_kwargs={},
    )

## Timeseries of radiometry and elevation

In [ ]:
static_nrb_mean_db = ds_static_nrb.HH_gamma0_db.mean(dim=["y", "x"], skipna=True)
timseries_nrb_mean_db = ds_timeseries_nrb.HH_gamma0_db.mean(dim=["y", "x"], skipna=True)
static_elevation_mean = ds_static_dem.elevation.mean(dim=["y", "x"])
timeseries_elevation_mean = ds_timeseries_dem.elevation.mean(dim=["y", "x"])
static_slope_mean = ds_static_dem.slope.mean(dim=["y", "x"])
timeseries_slope_mean = ds_timeseries_dem.slope.mean(dim=["y", "x"])
static_aspect_mean = ds_static_dem.aspect.mean(dim=["y", "x"])
timeseries_aspect_mean = ds_timeseries_dem.aspect.mean(dim=["y", "x"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

# --- Left: NRB mean comparison ---
print(f'PLotting NRB')
axes[0][0].plot(static_nrb_mean_db.time, static_nrb_mean_db, marker='o', label='Static NRB')
axes[0][0].plot(timseries_nrb_mean_db.time, timseries_nrb_mean_db, marker='x', label='Timeseries NRB')
axes[0][0].set_title("NRB Mean Backscatter Over Time")
axes[0][0].set_xlabel("Time")
axes[0][0].set_ylabel("HH_gamma0 (dB)")
axes[0][0].legend()
axes[0][0].grid(True)

# --- Right: elevation mean comparison ---
print(f'PLotting elevation')
axes[0][1].plot(static_elevation_mean.time, static_elevation_mean, marker='o', label='Static DEM')
axes[0][1].plot(timeseries_elevation_mean.time, timeseries_elevation_mean, marker='x', label='Timeseries DEM')
axes[0][1].set_title("DEM Mean Elevation Over Time")
axes[0][1].set_xlabel("Time")
axes[0][1].set_ylabel("Elevation (m)")
axes[0][1].legend()
axes[0][1].grid(True)

# --- Right: slope mean comparison ---
print(f'Plotting slope')
axes[1][0].plot(static_slope_mean.time, static_slope_mean, marker='o', label='Static DEM')
axes[1][0].plot(timeseries_slope_mean.time, timeseries_slope_mean, marker='x', label='Timeseries DEM')
axes[1][0].set_title("DEM Mean Slope Over Time")
axes[1][0].set_xlabel("Time")
axes[1][0].set_ylabel("Slope (degrees)")
axes[1][0].legend()
axes[1][0].grid(True)

# --- Right: aspect mean comparison ---
print(f'Plotting aspect')
axes[1][1].plot(static_aspect_mean.time, static_aspect_mean, marker='o', label='Static DEM')
axes[1][1].plot(timeseries_aspect_mean.time, timeseries_aspect_mean, marker='x', label='Timeseries DEM')
axes[1][1].set_title("DEM Mean Aspect Over Time")
axes[1][1].set_xlabel("Time")
axes[1][1].set_ylabel("Slope (degrees)")
axes[1][1].legend()
axes[1][1].grid(True)

plt.suptitle(f'Burst ID : {burst}')
plt.tight_layout()
plt.show()

## Plot timeseries max difference maps

In [ ]:
# Compute difference: last timestep minus first timestep
for param in ['elevation','slope','aspect']:
    diff = ds_timeseries_dem[param].isel(time=-2) - ds_timeseries_dem[param].isel(time=1)
    plt.figure(figsize=(12, 5))
    diff.plot.imshow(cmap="RdBu")
    plt.title(f"{param.upper()} Difference: 2023 minus 2015")
    plt.show()

# Combine GIFS

In [ ]:
def uniform_frames(gif_path, frame_interval=100):
    """Convert GIF to a list of frames at uniform intervals (ms). As there are
    'empty' repeated frames in the winter months of the .gif, we need to ensure
    the frames and intervals are consistent"""
    gif = Image.open(gif_path)
    frames_list = [frame.convert("RGBA") for frame in ImageSequence.Iterator(gif)]
    
    # Original frame durations
    durations = [frame.info.get('duration', gif.info['duration']) for frame in ImageSequence.Iterator(gif)]
    cum_times = np.cumsum(durations)
    total_time = cum_times[-1]
    
    # Number of uniform frames
    times = np.arange(0, total_time, frame_interval)
    
    uniform_frames = []
    f_idx = 0
    for t in times:
        # Advance to correct frame for this time
        while f_idx < len(cum_times) - 1 and t >= cum_times[f_idx]:
            f_idx += 1
        uniform_frames.append(frames_list[f_idx])
    
    return uniform_frames

combine = False
if combine:

    # Load uniform frames
    frames1 = uniform_frames(f"{burst}_REMA_10_gamma0.gif", frame_interval=80)
    frames2 = uniform_frames(f"{burst}_REMA_10_TIMESERIES_gamma0.gif", frame_interval=80)

    # Combine side by side
    combined_frames = []
    for f1, f2 in zip(frames1, frames2):
        new_frame = Image.new("RGBA", (f1.width + f2.width, f1.height))
        new_frame.paste(f1, (0, 0))
        new_frame.paste(f2, (f1.width, 0))
        combined_frames.append(new_frame)

    # Save combined GIF
    combined_frames[0].save(
        f"{burst}_gamma0_comparison.gif",
        save_all=True,
        append_images=combined_frames[1:],
        duration=80,  # uniform frame duration
        loop=0
    )
    combined_frames[0]